# Coleta de dados da Steam

Este notebook monta o catálogo de jogos, acrescenta descrições, gêneros e tags e coleta uma amostra de avaliações para o projeto de recomendação com PLN.

## Preparação e execução

1. Execute a célula de instalação abaixo para instalar as dependências no ambiente do kernel.
2. Configure `STEAM_API_KEY` no ambiente ou em `1_Coleta_Dados/.env`, seguindo `.env.example`.
3. Execute as células de cima para baixo. A listagem, o enriquecimento e a coleta de avaliações fazem requisições e gravam arquivos; não são apenas definições de funções.
4. Para usar o fallback de tags com Selenium, disponibilize o Chrome e o ambiente necessário ao WebDriver.

Os caminhos são resolvidos a partir da raiz do repositório, mesmo quando o kernel inicia em uma subpasta. Após alterar configurações, reinicie o kernel e execute novamente desde o início.

| Arquivo | Finalidade |
| --- | --- |
| `_DadosBrutos/jogos_steam.csv` | Catálogo e informações enriquecidas dos jogos |
| `_DadosBrutos/steam_reviews.csv` | Amostra de avaliações |
| `_DadosBrutos/steam_reviews.csv.state.json` | Pares de jogo/tipo concluídos para retomada |
| `1_Coleta_Dados/steam_rate_limit_state.json` | Contador local de chamadas da Web API por dia UTC |

### Instalação de dependências

O comando `%pip` instala as bibliotecas no ambiente do kernel e pode precisar de acesso à internet. Se houver solicitação de reinicialização, reinicie o kernel antes de continuar.


In [ ]:
%pip install pandas requests selenium python-dotenv


### 1. Caminhos e credenciais

Localiza o repositório, cria a pasta de dados brutos e configura a importação de `steam_scraper_support.py`. Variáveis de ambiente já definidas têm prioridade sobre o arquivo `.env`.


In [1]:
from pathlib import Path

# Resolve o repositorio a partir da raiz ou de qualquer subpasta.
raiz_projeto = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "1_Coleta_Dados").is_dir()
     and (p / "2_Limpeza_Preparacao").is_dir()),
    None,
)
if raiz_projeto is None:
    raise RuntimeError("Execute o notebook dentro do repositorio PLN_SteamRecommend.")

import sys
pasta_coleta = raiz_projeto / "1_Coleta_Dados"
pasta_brutos = raiz_projeto / "_DadosBrutos"
pasta_brutos.mkdir(parents=True, exist_ok=True)
if str(pasta_coleta) not in sys.path:
    sys.path.insert(0, str(pasta_coleta))

import os
from dotenv import load_dotenv

# Existing environment variables (including Colab secrets) take precedence.
load_dotenv(dotenv_path=pasta_coleta / ".env", override=False)
token = os.getenv("STEAM_API_KEY", "")
urlJogos = "https://api.steampowered.com"
urlLoja = "https://store.steampowered.com"

## 2. Dependências

Importa as bibliotecas de manipulação de dados, acesso HTTP, concorrência e tratamento de texto. Selenium é usado pelo fallback de extração de tags; sua ausência é informada na saída da célula.


In [2]:
import json
import re
import unicodedata
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from html import unescape
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd
import requests

try:
    from selenium import webdriver
    from selenium.common.exceptions import NoSuchElementException, TimeoutException, WebDriverException
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import Select, WebDriverWait
except ModuleNotFoundError:
    webdriver = None
    print("Selenium nao esta instalado. Rode: pip install selenium")

## 3. Controle de requisições e funções auxiliares

O limitador local usa um orçamento configurado de 100.000 chamadas por dia, com margem de 500, para as chamadas que recebem esse limitador. Ao esgotar o orçamento, a configuração padrão aguarda a virada do dia em UTC. Esse contador não representa chamadas feitas por outros programas.

O módulo de suporte concentra tentativas HTTP, controle de espera e gravação de CSV por arquivo temporário seguido de substituição. O checkpoint das avaliações é salvo depois dos dados. As funções de normalização tratam textos e problemas de codificação; `normalizar_csv_existente` regrava o arquivo recebido quando chamada.


In [3]:
STEAM_WEB_API_DAILY_LIMIT = 100_000
STEAM_WEB_API_SAFETY_MARGIN = 500
RATE_LIMIT_STATE_FILE = pasta_coleta / "steam_rate_limit_state.json"


class SteamDailyRateLimiter:
    """Persistent daily counter for Steam Web API calls."""

    def __init__(
        self,
        limit=STEAM_WEB_API_DAILY_LIMIT,
        safety_margin=STEAM_WEB_API_SAFETY_MARGIN,
        state_file=RATE_LIMIT_STATE_FILE,
        wait_when_exhausted=True,
    ):
        self.limit = limit
        self.safety_margin = safety_margin
        self.state_file = Path(state_file)
        self.wait_when_exhausted = wait_when_exhausted
        self.state = self._load_state()
        self._reset_if_needed()

    @property
    def usable_limit(self):
        return max(0, self.limit - self.safety_margin)

    @staticmethod
    def _today_utc():
        return datetime.now(timezone.utc).date().isoformat()

    @staticmethod
    def _seconds_until_tomorrow_utc():
        now = datetime.now(timezone.utc)
        tomorrow = (now + timedelta(days=1)).date()
        reset_at = datetime.combine(tomorrow, datetime.min.time(), timezone.utc)
        return max(1, int((reset_at - now).total_seconds()) + 1)

    def _load_state(self):
        if not self.state_file.exists():
            return {"date_utc": self._today_utc(), "calls": 0}

        try:
            return json.loads(self.state_file.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError):
            return {"date_utc": self._today_utc(), "calls": 0}

    def _save_state(self):
        self.state_file.write_text(
            json.dumps(self.state, indent=2),
            encoding="utf-8",
        )

    def _reset_if_needed(self):
        today = self._today_utc()
        if self.state.get("date_utc") != today:
            self.state = {"date_utc": today, "calls": 0}
            self._save_state()

    def acquire(self, calls=1):
        self._reset_if_needed()

        if self.state["calls"] + calls > self.usable_limit:
            wait_seconds = self._seconds_until_tomorrow_utc()
            message = (
                "Steam Web API daily request budget reached "
                f"({self.state['calls']}/{self.usable_limit})."
            )

            if not self.wait_when_exhausted:
                raise RuntimeError(message)

            print(f"{message} Waiting {wait_seconds // 60} minutes for the UTC reset...")
            time.sleep(wait_seconds)
            self._reset_if_needed()

        self.state["calls"] += calls
        self._save_state()


steam_web_api_limiter = SteamDailyRateLimiter()


from steam_scraper_support import (
    steam_get, atomic_csv, REVIEW_COLUMNS,
    load_review_checkpoint, save_review_checkpoint,
)


def criar_session_steam(pool_size=32):
    session = requests.Session()
    adapter = requests.adapters.HTTPAdapter(
        pool_connections=pool_size,
        pool_maxsize=pool_size,
        max_retries=0,
    )
    session.mount("https://", adapter)
    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
    })
    session.cookies.update({
        "birthtime": "568022401",
        "lastagecheckage": "1-January-1988",
        "wants_mature_content": "1",
    })
    return session


def normalizar_texto(valor):
    if pd.isna(valor):
        return ""

    texto = str(valor)
    sinais_mojibake = ["\u00c3", "\u00c2", "\u00e2\u20ac", "\u00e2\u201e", "\ufffd"]

    def pontuar_mojibake(texto_avaliado):
        return sum(texto_avaliado.count(sinal) for sinal in sinais_mojibake)

    if pontuar_mojibake(texto) > 0:
        try:
            texto_corrigido = texto.encode("cp1252").decode("utf-8")
            if "\ufffd" not in texto_corrigido and pontuar_mojibake(texto_corrigido) < pontuar_mojibake(texto):
                texto = texto_corrigido
        except UnicodeError:
            pass

    texto = unicodedata.normalize("NFC", texto)
    texto = "".join(
        caractere
        for caractere in texto
        if caractere.isprintable() or caractere in "\n\t"
    )
    texto = " ".join(texto.split())

    return texto.strip()


def normalizar_csv_existente(arquivo_csv):
    df = pd.read_csv(arquivo_csv, encoding="utf-8-sig")

    for coluna in df.select_dtypes(include="object").columns:
        df[coluna] = df[coluna].apply(normalizar_texto)

    df.to_csv(arquivo_csv, index=False, encoding="utf-8-sig")
    print(f"Arquivo '{arquivo_csv}' normalizado.")

    return df

## 4. Listagem do catálogo

Consulta a lista de aplicativos com filtros para jogos, pagina por `appid`, remove nomes vazios e duplicatas e salva o catálogo. A chamada ao final desta célula inicia a coleta.

- `quantidade=None`: consulta todos os jogos retornados; um inteiro limita a coleta.
- `tamanho_pagina=50_000`: máximo solicitado por página, limitado pelo código a 50.000.
- `intervalo=1.0`: espera configurada entre consultas.
- `arquivo_saida`: destino do catálogo em `_DadosBrutos`.

Quando o CSV já existe, os metadados recebidos são combinados com os registros anteriores, preservando informações de enriquecimento e entradas antigas. Os campos básicos são `appid`, `name`, `last_modified` e `price_change_number`, quando retornados pela API.


In [ ]:
def listar_jogos_steam(
    quantidade=None,
    arquivo_saida=pasta_brutos / "jogos_steam.csv",
    tamanho_pagina=50_000,
    intervalo=1.0,
):
    """
    Lista jogos da Steam usando paginacao oficial.

    Use quantidade=None para coletar todos os jogos publicos retornados pela API.
    """
    if not token:
        raise ValueError("Set the STEAM_API_KEY environment variable before listing games.")
    url = f"{urlJogos}/IStoreService/GetAppList/v1/"
    todos_aplicativos = []
    ultimo_appid = None

    while True:
        limite_pagina = min(tamanho_pagina, 50_000)
        if quantidade is not None:
            restantes = quantidade - len(todos_aplicativos)
            if restantes <= 0:
                break
            limite_pagina = min(limite_pagina, restantes)

        parametros = {
            "key": token,
            "include_games": "true",
            "include_dlc": "false",
            "include_software": "false",
            "include_videos": "false",
            "include_hardware": "false",
            "max_results": limite_pagina,
        }

        if ultimo_appid is not None:
            parametros["last_appid"] = ultimo_appid

        response = steam_get(
            url,
            params=parametros,
            timeout=60,
            limiter=steam_web_api_limiter,
            intervalo=intervalo,
        )

        aplicativos = response.json().get("response", {}).get("apps", [])
        if not aplicativos:
            break

        todos_aplicativos.extend(aplicativos)
        ultimo_appid = aplicativos[-1]["appid"]

        print(f"{len(todos_aplicativos)} jogos carregados ate appid {ultimo_appid}.")

        if len(aplicativos) < limite_pagina:
            break

        time.sleep(intervalo)

    df = pd.DataFrame(todos_aplicativos)

    if df.empty:
        df = pd.DataFrame(columns=["appid", "name", "last_modified", "price_change_number"])
    else:
        colunas = [col for col in ["appid", "name", "last_modified", "price_change_number"] if col in df.columns]
        df = df[colunas].copy()
        df["name"] = df["name"].apply(normalizar_texto)
        df = df[df["name"] != ""]
        df = (
            df
            .drop_duplicates(subset="appid")
            .sort_values("appid")
            .reset_index(drop=True)
        )

    if Path(arquivo_saida).exists():
        existente = pd.read_csv(arquivo_saida, encoding="utf-8-sig")
        if "appid" not in existente:
            raise ValueError("Existing catalog is missing appid")
        existente = existente.drop_duplicates("appid").set_index("appid")
        novos = df.set_index("appid")
        # Refresh catalog metadata while preserving enrichment and older entries.
        df = novos.combine_first(existente).reset_index().sort_values("appid")
    atomic_csv(df, arquivo_saida)
    print(f"{len(df)} jogos salvos em '{arquivo_saida}'.")

    return df


jogos_df = listar_jogos_steam()
print(jogos_df.head(10))

50000 jogos carregados ate appid 1538580.
100000 jogos carregados ate appid 2804430.
150000 jogos carregados ate appid 4117160.
186355 jogos carregados ate appid 5251700.
186406 jogos salvos em 'jogos_steam.csv'.
   appid                            name  last_modified  price_change_number  \
0     10                  Counter-Strike     1745368572             37149137   
1     20           Team Fortress Classic     1745368565             37149137   
2     30                   Day of Defeat     1745368580             37149137   
3     40              Deathmatch Classic     1745368570             37149137   
4     50       Half-Life: Opposing Force     1745368539             37149137   
5     60                        Ricochet     1745368568             37149137   
6     70                       Half-Life     1745368462             37149137   
7     80  Counter-Strike: Condition Zero     1745368574             37149137   
8    130           Half-Life: Blue Shift     1745368541            

## 5. Funções de enriquecimento

Define as versões sequenciais e concorrentes para buscar descrição, gêneros e tags. As informações da loja são solicitadas em inglês, com região `us`. As tags são extraídas da página da loja, com possibilidade de fallback via Selenium.

As funções atualizam o próprio CSV do catálogo. `pular_existentes` permite aproveitar informações já presentes; `salvar_a_cada` controla a frequência de gravação durante o processamento.


In [ ]:
def buscar_informacoes_jogo(appid, intervalo=1.0, session=None):
    url = f"{urlLoja}/api/appdetails"

    parametros = {
        "appids": appid,
        "l": "english",
        "cc": "us",
    }

    response = steam_get(
        url,
        params=parametros,
        timeout=30,
        intervalo=intervalo,
        session=session,
    )

    resultado = response.json().get(str(appid), {})

    if not resultado.get("success"):
        return None

    dados = resultado["data"]

    return {
        "description": normalizar_texto(dados.get("short_description", "")),
        "genres": "; ".join(
            normalizar_texto(genre.get("description", ""))
            for genre in dados.get("genres", [])
            if genre.get("description")
        ),
    }


def atualizar_csv_com_informacoes(
    arquivo_csv=pasta_brutos / "jogos_steam.csv",
    intervalo=1.0,
    salvar_a_cada=50,
    pular_existentes=True,
):
    df = pd.read_csv(arquivo_csv)

    if not {"appid", "name"}.issubset(df.columns):
        raise ValueError("O CSV precisa ter as colunas 'appid' e 'name'.")

    for coluna in ["description", "genres"]:
        if coluna not in df.columns:
            df[coluna] = ""
        df[coluna] = df[coluna].apply(normalizar_texto)

    total = len(df)

    for indice, linha in df.iterrows():
        if pular_existentes and str(linha.get("description", "")).strip():
            continue

        appid = int(linha["appid"])
        nome = normalizar_texto(linha["name"])

        print(f"[{indice + 1}/{total}] Buscando {nome}...")

        try:
            informacoes = buscar_informacoes_jogo(appid, intervalo=intervalo)

            if informacoes is None:
                print("  Informacoes nao disponiveis.")
                continue

            df.at[indice, "description"] = informacoes["description"]
            df.at[indice, "genres"] = informacoes["genres"]

        except requests.RequestException as erro:
            print(f"  Erro na requisicao: {erro}")

        except (KeyError, ValueError) as erro:
            print(f"  Erro ao processar resposta: {erro}")

        if (indice + 1) % salvar_a_cada == 0:
            atomic_csv(df, arquivo_csv)

        time.sleep(intervalo)

    atomic_csv(df, arquivo_csv)
    print(f"Arquivo '{arquivo_csv}' atualizado.")

    return df



def atualizar_csv_com_informacoes_rapido(
    arquivo_csv=pasta_brutos / "jogos_steam.csv",
    workers=12,
    salvar_a_cada=500,
    pular_existentes=True,
    intervalo_requisicoes=0.75,
    max_jogos=None,
    progresso_a_cada=50,
):
    from steam_scraper_support import gate_for
    if workers < 1 or salvar_a_cada < 1 or progresso_a_cada < 1:
        raise ValueError("Workers e frequencias devem ser positivos.")
    if max_jogos is not None and (not isinstance(max_jogos, int) or max_jogos < 1):
        raise ValueError("max_jogos deve ser um inteiro positivo.")
    gate_for(urlLoja).configure(intervalo_requisicoes)
    inicio = time.perf_counter()
    df = pd.read_csv(arquivo_csv, encoding="utf-8-sig")

    if not {"appid", "name"}.issubset(df.columns):
        raise ValueError("O CSV precisa ter as colunas 'appid' e 'name'.")

    for coluna in ["description", "genres"]:
        if coluna not in df.columns:
            df[coluna] = ""
        df[coluna] = df[coluna].apply(normalizar_texto)

    pendentes = [
        (indice, int(linha["appid"]), normalizar_texto(linha["name"]))
        for indice, linha in df.iterrows()
        if not (pular_existentes and str(linha.get("description", "")).strip())
    ]

    if max_jogos is not None:
        pendentes = pendentes[:max_jogos]

    total = len(pendentes)
    print(f"Coletando informacoes de {total} jogos com {workers} workers...")

    if total == 0:
        return df

    session = criar_session_steam(pool_size=workers * 2)
    processados = 0
    com_descricao = 0
    erros = 0
    segundos_salvando = 0.0

    def tarefa(item):
        indice, appid, nome = item
        informacoes = buscar_informacoes_jogo(appid, session=session)
        return indice, appid, nome, informacoes

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futuros = [executor.submit(tarefa, item) for item in pendentes]

        for futuro in as_completed(futuros):
            processados += 1

            try:
                indice, appid, nome, informacoes = futuro.result()
                if informacoes is not None:
                    com_descricao += bool(informacoes["description"].strip())
                    df.at[indice, "description"] = informacoes["description"]
                    df.at[indice, "genres"] = informacoes["genres"]
            except requests.RequestException as erro:
                erros += 1
                print(f"  Erro na requisicao: {erro}")
            except (KeyError, ValueError) as erro:
                erros += 1
                print(f"  Erro ao processar resposta: {erro}")

            if processados % salvar_a_cada == 0 or processados == total:
                inicio_salvamento = time.perf_counter()
                atomic_csv(df, arquivo_csv)
                segundos_salvando += time.perf_counter() - inicio_salvamento
            if processados % progresso_a_cada == 0 or processados == total:
                decorrido = time.perf_counter() - inicio
                print(f"  {processados}/{total}: {processados * 60 / decorrido:.1f} jogos/min; "
                      f"{com_descricao} com descricao; {erros} erros; {decorrido:.1f}s", flush=True)

    session.close()
    print(f"Arquivo '{arquivo_csv}' atualizado.")
    df.attrs["benchmark"] = {
        "processed": processados, "with_description": com_descricao, "errors": erros,
        "elapsed_seconds": time.perf_counter() - inicio,
        "save_seconds": segundos_salvando, "request_interval": intervalo_requisicoes,
    }
    return df

def criar_driver_selenium(headless=True, page_load_timeout=30):
    if webdriver is None:
        raise ModuleNotFoundError("Instale o Selenium com: pip install selenium")

    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1366,900")
    options.add_argument("--lang=en-US")
    options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(page_load_timeout)
    configurar_cookies_steam(driver)
    return driver


def configurar_cookies_steam(driver):
    driver.get(f"{urlLoja}/?l=english")
    cookies = [
        {"name": "birthtime", "value": "568022401"},
        {"name": "lastagecheckage", "value": "1-January-1988"},
        {"name": "wants_mature_content", "value": "1"},
    ]

    for cookie in cookies:
        driver.add_cookie(cookie)


def passar_age_gate_se_necessario(driver):
    try:
        seletor_ano = driver.find_element(By.ID, "ageYear")
        Select(seletor_ano).select_by_value("1988")
        driver.find_element(By.ID, "view_product_page_btn").click()
    except NoSuchElementException:
        return


def limpar_tags(tags):
    tags_limpas = []
    for tag in tags:
        tag = normalizar_texto(tag)
        if tag and tag != "+" and tag not in tags_limpas:
            tags_limpas.append(tag)

    return tags_limpas


def buscar_tags_jogo_selenium(driver, appid, espera=10):
    url = f"{urlLoja}/app/{appid}/?l=english&cc=us"
    driver.get(url)
    passar_age_gate_se_necessario(driver)

    try:
        WebDriverWait(driver, espera).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a.app_tag"))
        )
    except TimeoutException:
        return []

    elementos = driver.find_elements(
        By.CSS_SELECTOR,
        ".glance_tags.popular_tags a.app_tag, a.app_tag",
    )

    return limpar_tags([elemento.text for elemento in elementos])



def extrair_tags_do_html(html_text):
    bloco = re.search(
        r'<div[^>]+class="[^"]*glance_tags\s+popular_tags[^"]*"[^>]*>(.*?)</div>',
        html_text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    tags_html = bloco.group(1) if bloco else html_text

    tags = re.findall(
        r'<a[^>]+class="[^"]*app_tag[^"]*"[^>]*>(.*?)</a>',
        tags_html,
        flags=re.IGNORECASE | re.DOTALL,
    )

    return limpar_tags([
        unescape(re.sub(r'<[^>]+>', '', tag))
        for tag in tags
    ])


def buscar_tags_jogo_requests(appid, session=None, intervalo=1.0):
    url = f"{urlLoja}/app/{appid}/"
    response = steam_get(
        url,
        params={"l": "english", "cc": "us"},
        timeout=30,
        intervalo=intervalo,
        session=session,
    )
    return extrair_tags_do_html(response.text)


def atualizar_csv_com_tags_rapido(
    arquivo_csv=pasta_brutos / "jogos_steam.csv",
    workers=12,
    salvar_a_cada=500,
    pular_existentes=True,
    usar_selenium_fallback=True,
    headless=True,
):
    df = pd.read_csv(arquivo_csv, encoding="utf-8-sig")

    if not {"appid", "name"}.issubset(df.columns):
        raise ValueError("O CSV precisa ter as colunas 'appid' e 'name'.")

    if "user_tags" not in df.columns:
        df["user_tags"] = ""

    df["user_tags"] = df["user_tags"].apply(normalizar_texto)
    pendentes = [
        (indice, int(linha["appid"]), normalizar_texto(linha["name"]))
        for indice, linha in df.iterrows()
        if not (pular_existentes and str(linha.get("user_tags", "")).strip())
    ]

    total = len(pendentes)
    print(f"Coletando tags por requests de {total} jogos com {workers} workers...")

    if total == 0:
        return df

    session = criar_session_steam(pool_size=workers * 2)
    falhas_para_selenium = []
    processados = 0

    def tarefa(item):
        indice, appid, nome = item
        tags = buscar_tags_jogo_requests(appid, session=session)
        return indice, appid, nome, tags

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futuros = {executor.submit(tarefa, item): item for item in pendentes}

        for futuro in as_completed(futuros):
            processados += 1

            try:
                indice, appid, nome, tags = futuro.result()
                if tags:
                    df.at[indice, "user_tags"] = "; ".join(tags)
                else:
                    falhas_para_selenium.append((indice, appid, nome))
            except requests.RequestException as erro:
                falhas_para_selenium.append(futuros[futuro])
                print(f"  Erro na requisicao de tags: {erro}")
            except (KeyError, ValueError) as erro:
                falhas_para_selenium.append(futuros[futuro])
                print(f"  Erro ao processar tags: {erro}")

            if processados % salvar_a_cada == 0 or processados == total:
                atomic_csv(df, arquivo_csv)
                print(f"  {processados}/{total} tags por requests processadas.")

    session.close()

    if usar_selenium_fallback and falhas_para_selenium:
        print(f"Usando Selenium como fallback em {len(falhas_para_selenium)} jogos sem tags.")
        driver = criar_driver_selenium(headless=headless)

        try:
            for posicao, (indice, appid, nome) in enumerate(falhas_para_selenium, start=1):
                try:
                    from steam_scraper_support import gate_for
                    gate_for(urlLoja).acquire()
                    tags = buscar_tags_jogo_selenium(driver, appid)
                    df.at[indice, "user_tags"] = "; ".join(tags)
                    print(f"  Selenium {posicao}/{len(falhas_para_selenium)}: {nome} ({len(tags)} tags)")
                except WebDriverException as erro:
                    print(f"  Erro no Selenium: {erro}")

                if posicao % salvar_a_cada == 0 or posicao == len(falhas_para_selenium):
                    atomic_csv(df, arquivo_csv)
        finally:
            driver.quit()

    atomic_csv(df, arquivo_csv)
    print(f"Arquivo '{arquivo_csv}' atualizado com tags.")
    return df

def atualizar_csv_com_tags(
    arquivo_csv=pasta_brutos / "jogos_steam.csv",
    intervalo=1.5,
    salvar_a_cada=50,
    pular_existentes=True,
    headless=True,
):
    df = pd.read_csv(arquivo_csv)

    if not {"appid", "name"}.issubset(df.columns):
        raise ValueError("O CSV precisa ter as colunas 'appid' e 'name'.")

    if "user_tags" not in df.columns:
        df["user_tags"] = ""

    df["user_tags"] = df["user_tags"].apply(normalizar_texto)
    total = len(df)

    driver = criar_driver_selenium(headless=headless)

    try:
        for indice, linha in df.iterrows():
            if pular_existentes and str(linha.get("user_tags", "")).strip():
                continue

            appid = int(linha["appid"])
            nome = normalizar_texto(linha["name"])

            print(f"[{indice + 1}/{total}] Raspando tags: {nome}...")

            try:
                tags = buscar_tags_jogo_selenium(driver, appid)
                df.at[indice, "user_tags"] = "; ".join(tags)
                print(f"  {len(tags)} tags encontradas.")

            except WebDriverException as erro:
                print(f"  Erro no Selenium: {erro}")

            if (indice + 1) % salvar_a_cada == 0:
                atomic_csv(df, arquivo_csv)

            time.sleep(intervalo)

    finally:
        driver.quit()

    atomic_csv(df, arquivo_csv)
    print(f"Arquivo '{arquivo_csv}' atualizado com tags.")

    return df

## 6. Execução do enriquecimento

Atualiza descrições e gêneros e, em seguida, tags. A configuração desta célula usa 12 workers, grava a cada 500 itens e pula informações já existentes. O fallback com Selenium está habilitado em modo headless, sem janela visível do navegador.

A saída mostra o andamento e uma prévia das colunas `appid`, `name`, `genres`, `description` e `user_tags`. O destino é `_DadosBrutos/jogos_steam.csv`.


In [ ]:
jogos_df = atualizar_csv_com_informacoes_rapido(
    arquivo_csv=pasta_brutos / "jogos_steam.csv",
    workers=12,
    salvar_a_cada=500,
    pular_existentes=True,
)

jogos_df = atualizar_csv_com_tags_rapido(
    arquivo_csv=pasta_brutos / "jogos_steam.csv",
    workers=12,
    salvar_a_cada=500,
    pular_existentes=True,
    usar_selenium_fallback=True,
    headless=True,
)

print(jogos_df[["appid", "name", "genres", "description", "user_tags"]].head(10))


Coletando informacoes de 461 jogos com 12 workers...
  50/461: 70.4 jogos/min; 0 com descricao; 0 erros; 42.6s


HTTP 500 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)


  100/461: 69.8 jogos/min; 0 com descricao; 0 erros; 85.9s


HTTP 500 from store.steampowered.com: shared cooldown 10.0s (attempt 2/5)
HTTP 500 from store.steampowered.com: shared cooldown 20.0s (attempt 3/5)
HTTP 500 from store.steampowered.com: shared cooldown 40.0s (attempt 4/5)
HTTP 500 from store.steampowered.com: shared cooldown 80.0s (attempt 5/5)


  Erro na requisicao: 500 Server Error: Internal Server Error for url: https://store.steampowered.com/api/appdetails?appids=1196310&l=english&cc=us
  150/461: 32.8 jogos/min; 2 com descricao; 1 erros; 274.6s
  200/461: 38.5 jogos/min; 3 com descricao; 1 erros; 311.7s
  250/461: 43.0 jogos/min; 3 com descricao; 1 erros; 349.2s
  300/461: 46.5 jogos/min; 9 com descricao; 1 erros; 386.7s
  350/461: 49.5 jogos/min; 20 com descricao; 1 erros; 424.3s


HTTP 429 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 429 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 429 from store.steampowered.com: shared cooldown 10.0s (attempt 2/5)
HTTP 429 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 429 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 429 from store.steampowered.com: shared cooldown 10.0s (attempt 2/5)
HTTP 429 from store.steampowered.com: shared cooldown 10.0s (attempt 2/5)
HTTP 429 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 429 from store.steampowered.com: shared cooldown 10.0s (attempt 2/5)
HTTP 429 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 429 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 429 from store.steampowered.com: shared cooldown 20.0s (attempt 3/5)
HTTP 429 from store.steampowered.com: shared cooldown 40.0s (attempt 4/5)
HTTP 429 from store.steampowered.com: shared 

  Erro na requisicao: 429 Client Error: Too Many Requests for url: https://store.steampowered.com/api/appdetails?appids=5171980&l=english&cc=us
  400/461: 35.3 jogos/min; 57 com descricao; 2 erros; 680.2s
  450/461: 37.6 jogos/min; 107 com descricao; 2 erros; 717.7s
  461/461: 38.1 jogos/min; 118 com descricao; 2 erros; 726.8s
Arquivo 'jogos_steam.csv' atualizado.
Coletando tags por requests de 365 jogos com 12 workers...
  365/365 tags por requests processadas.
Usando Selenium como fallback em 247 jogos sem tags.
  Selenium 1/247: Championship Manager 2007 (0 tags)
  Selenium 2/247: The Cursed Crusade (0 tags)
  Selenium 3/247: Fallout: New Vegas (0 tags)
  Selenium 4/247: Freeze Tag Fun Pack #1 (0 tags)
  Selenium 5/247: Dishonored (0 tags)
  Selenium 6/247: AION MMO (0 tags)
  Selenium 7/247: Max Payne (0 tags)
  Selenium 8/247: Sniper Elite: Zombie Army (0 tags)
  Selenium 9/247: Max Payne (0 tags)
  Selenium 10/247: NOBUNAGA'S AMBITION: Souzou with Power Up Kit (0 tags)
  Selenium

## 7. Regras de coleta das avaliações

Define a coleta sequencial e concorrente. Um jogo precisa ter pelo menos 100 avaliações no total, considerando todos os idiomas e os dois tipos, para entrar na amostra. Para cada jogo elegível, são solicitadas avaliações recentes em inglês: até 10 positivas e até 10 negativas. A quantidade efetivamente retornada pode ser menor; a amostra não representa todas as avaliações da Steam.

`quantidade_por_tipo` aceita valores de 1 a 10. Com `pular_existentes=True`, o CSV e seu checkpoint permitem retomar pares `(appid, review_type)` já concluídos, inclusive consultas bem-sucedidas sem resultados. Erros não devem marcar o par como concluído. Use a retomada para dados produzidos com as mesmas regras; o padrão das funções é reconstruir a coleta.

Cada linha contém a identificação do jogo, o tipo e texto da avaliação, `voted_up`, data de criação, votos de utilidade e humor, `weighted_vote_score` e tempo de jogo informado pela API. O JSON de estado deve permanecer ao lado do CSV correspondente.


In [ ]:
def buscar_reviews_jogo(appid, review_type, quantidade=10, intervalo=1.0, session=None):
    """Return up to 10 English reviews for games with at least 100 total reviews."""
    if review_type not in ("positive", "negative"):
        raise ValueError("review_type must be positive or negative")
    if not 1 <= quantidade <= 10:
        raise ValueError("quantidade must be between 1 and 10 per sentiment")
    url = f"{urlLoja}/appreviews/{appid}"

    # Check both sentiments and all languages before fetching the sample.
    resumo = steam_get(
        url,
        params={"json": 1, "language": "all", "filter": "recent",
                "review_type": "all", "purchase_type": "all",
                "num_per_page": 1, "cursor": "*"},
        timeout=30,
        intervalo=intervalo,
        session=session,
    ).json()
    if resumo.get("success") != 1:
        raise ValueError(f"Steam review summary unsuccessful for {appid}")
    if int(resumo["query_summary"]["total_reviews"]) < 100:
        return []

    parametros = {
        "json": 1,
        "language": "english",
        "filter": "recent",
        "review_type": review_type,
        "purchase_type": "all",
        "num_per_page": quantidade,
    }

    response = steam_get(
        url,
        params=parametros,
        timeout=30,
        intervalo=intervalo,
        session=session,
    )

    dados = response.json()

    if dados.get("success") != 1:
        raise ValueError(f"Steam review request unsuccessful for {appid}")

    return dados.get("reviews", [])[:quantidade]



def coletar_reviews_rapido(
    arquivo_jogos=pasta_brutos / "jogos_steam.csv",
    arquivo_saida=pasta_brutos / "steam_reviews.csv",
    quantidade_por_tipo=10,
    workers=12,
    salvar_a_cada=500,
    pular_existentes=False,
):
    """Collect up to 10 reviews per sentiment from games with 100+ total reviews.

    Rebuild by default to discard legacy, unfiltered results. Only enable
    pular_existentes for checkpoints produced with this eligibility rule.
    """
    if not 1 <= quantidade_por_tipo <= 10:
        raise ValueError("quantidade_por_tipo must be between 1 and 10 per sentiment")
    jogos_df = pd.read_csv(arquivo_jogos, encoding="utf-8-sig")
    saida = Path(arquivo_saida)

    reviews_coletadas, chaves_processadas = load_review_checkpoint(
        saida, quantidade_por_tipo, pular_existentes,
    )

    tarefas = []
    for _, jogo in jogos_df.iterrows():
        appid = int(jogo["appid"])
        nome_jogo = normalizar_texto(jogo["name"])

        for tipo in ["positive", "negative"]:
            chave = (appid, tipo)
            if pular_existentes and chave in chaves_processadas:
                continue
            tarefas.append((appid, nome_jogo, tipo))

    total = len(tarefas)
    print(f"Coletando reviews de {total} pares jogo/tipo com {workers} workers...")

    if total == 0:
        return pd.DataFrame(reviews_coletadas, columns=REVIEW_COLUMNS)

    session = criar_session_steam(pool_size=workers * 2)
    processados = 0

    def tarefa(item):
        appid, nome_jogo, tipo = item
        reviews = buscar_reviews_jogo(
            appid=appid,
            review_type=tipo,
            quantidade=quantidade_por_tipo,
            session=session,
        )
        linhas = []

        for review in reviews:
            linhas.append({
                "appid": appid,
                "game_name": normalizar_texto(nome_jogo),
                "review_type": tipo,
                "voted_up": review.get("voted_up"),
                "review": normalizar_texto(review.get("review", "")),
                "timestamp_created": review.get("timestamp_created"),
                "votes_up": review.get("votes_up", 0),
                "votes_funny": review.get("votes_funny", 0),
                "weighted_vote_score": review.get("weighted_vote_score", 0),
                "playtime_forever": review.get("author", {}).get("playtime_forever", 0),
            })

        return appid, tipo, linhas

    with ThreadPoolExecutor(max_workers=workers) as executor:
        futuros = [executor.submit(tarefa, item) for item in tarefas]

        for futuro in as_completed(futuros):
            processados += 1

            try:
                appid, tipo, linhas = futuro.result()
                reviews_coletadas = [
                    row for row in reviews_coletadas
                    if (int(row["appid"]), row["review_type"]) != (appid, tipo)
                ]
                reviews_coletadas.extend(linhas)
                chaves_processadas.add((appid, tipo))
            except requests.RequestException as erro:
                print(f"  Erro na requisicao de review: {erro}")
            except (KeyError, ValueError) as erro:
                print(f"  Erro ao processar review: {erro}")

            if processados % salvar_a_cada == 0 or processados == total:
                save_review_checkpoint(reviews_coletadas, chaves_processadas,
                                       arquivo_saida, quantidade_por_tipo)
                print(f"  {processados}/{total} pares de reviews processados.")

    session.close()
    reviews_df = pd.DataFrame(reviews_coletadas, columns=REVIEW_COLUMNS)
    save_review_checkpoint(reviews_coletadas, chaves_processadas,
                           arquivo_saida, quantidade_por_tipo)
    print(f"\n{len(reviews_df)} reviews salvas em '{arquivo_saida}'.")
    return reviews_df

def coletar_reviews(
    arquivo_jogos=pasta_brutos / "jogos_steam.csv",
    arquivo_saida=pasta_brutos / "steam_reviews.csv",
    quantidade_por_tipo=10,
    intervalo=1.0,
    salvar_a_cada=50,
    pular_existentes=True,
):
    jogos_df = pd.read_csv(arquivo_jogos)
    saida = Path(arquivo_saida)

    reviews_coletadas, chaves_processadas = load_review_checkpoint(
        saida, quantidade_por_tipo, pular_existentes,
    )

    total = len(jogos_df)

    for indice, jogo in jogos_df.iterrows():
        appid = int(jogo["appid"])
        nome_jogo = normalizar_texto(jogo["name"])

        print(f"[{indice + 1}/{total}] Collecting reviews: {nome_jogo}")

        for tipo in ["positive", "negative"]:
            chave = (appid, tipo)
            if pular_existentes and chave in chaves_processadas:
                continue

            try:
                reviews = buscar_reviews_jogo(
                    appid=appid,
                    review_type=tipo,
                    quantidade=quantidade_por_tipo,
                    intervalo=intervalo,
                )

                reviews_coletadas = [
                    row for row in reviews_coletadas
                    if (int(row["appid"]), row["review_type"]) != chave
                ]
                for review in reviews:
                    reviews_coletadas.append({
                        "appid": appid,
                        "game_name": normalizar_texto(nome_jogo),
                        "review_type": tipo,
                        "voted_up": review.get("voted_up"),
                        "review": normalizar_texto(review.get("review", "")),
                        "timestamp_created": review.get("timestamp_created"),
                        "votes_up": review.get("votes_up", 0),
                        "votes_funny": review.get("votes_funny", 0),
                        "weighted_vote_score": review.get("weighted_vote_score", 0),
                        "playtime_forever": review.get("author", {}).get("playtime_forever", 0),
                    })

                chaves_processadas.add(chave)
                print(f"  {tipo}: {len(reviews)} reviews collected.")

            except (requests.RequestException, ValueError) as erro:
                print(f"  Request error ({tipo}): {erro}")

            time.sleep(intervalo)

        if (indice + 1) % salvar_a_cada == 0:
            save_review_checkpoint(reviews_coletadas, chaves_processadas,
                                       arquivo_saida, quantidade_por_tipo)

    reviews_df = pd.DataFrame(reviews_coletadas, columns=REVIEW_COLUMNS)
    save_review_checkpoint(reviews_coletadas, chaves_processadas,
                           arquivo_saida, quantidade_por_tipo)

    print(f"\n{len(reviews_df)} reviews saved in '{arquivo_saida}'.")
    return reviews_df

## 8. Execução e retomada da coleta

Esta chamada usa 10 avaliações por tipo, 12 workers e gravação a cada 500 pares processados. A retomada está habilitada explicitamente por `pular_existentes=True`.

O resultado é salvo em `_DadosBrutos/steam_reviews.csv`, com checkpoint no mesmo diretório. Confira os registros de erro e a prévia exibida antes de seguir para `2_Limpeza_Preparacao/PLN_LimpezaDados.ipynb`. As saídas armazenadas no notebook podem pertencer a execuções anteriores.


In [ ]:
reviews_df = coletar_reviews_rapido(
    arquivo_jogos=pasta_brutos / "jogos_steam.csv",
    arquivo_saida=pasta_brutos / "steam_reviews.csv",
    quantidade_por_tipo=10,
    workers=12,
    salvar_a_cada=500,
    pular_existentes=True,
)

print(reviews_df.head())


Coletando reviews de 195312 pares jogo/tipo com 12 workers...
  500/195312 pares de reviews processados.
  1000/195312 pares de reviews processados.
  1500/195312 pares de reviews processados.
  2000/195312 pares de reviews processados.
  2500/195312 pares de reviews processados.
  3000/195312 pares de reviews processados.


HTTP 502 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)


  3500/195312 pares de reviews processados.


HTTP 502 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)


  4000/195312 pares de reviews processados.
  4500/195312 pares de reviews processados.
  5000/195312 pares de reviews processados.
  5500/195312 pares de reviews processados.
  6000/195312 pares de reviews processados.
  6500/195312 pares de reviews processados.
  7000/195312 pares de reviews processados.
  Erro na requisicao de review: HTTPSConnectionPool(host='store.steampowered.com', port=443): Max retries exceeded with url: /appreviews/2602710?json=1&language=all&filter=recent&review_type=all&purchase_type=all&num_per_page=1&cursor=%2A (Caused by NameResolutionError("HTTPSConnection(host='store.steampowered.com', port=443): Failed to resolve 'store.steampowered.com' ([Errno 11001] getaddrinfo failed)"))
  Erro na requisicao de review: HTTPSConnectionPool(host='store.steampowered.com', port=443): Max retries exceeded with url: /appreviews/2602570?json=1&language=all&filter=recent&review_type=all&purchase_type=all&num_per_page=1&cursor=%2A (Caused by NameResolutionError("HTTPSConnec

HTTP 502 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 502 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)
HTTP 502 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)


  19000/195312 pares de reviews processados.


HTTP 502 from store.steampowered.com: shared cooldown 5.0s (attempt 1/5)


  19500/195312 pares de reviews processados.
  20000/195312 pares de reviews processados.
  20500/195312 pares de reviews processados.
  21000/195312 pares de reviews processados.
